In [1]:
import requests
import json
import timeit
import pickle
from multiprocessing import Pool
import os
import pandas as pd
from functools import partial

In [2]:
YEAR = 2022

API_BASE = f"https://fantasy.espncdn.com/tournament-challenge-bracket/{YEAR}/en/api/"

FIND_GROUP = "findGroups?type_access=0&type_group=0&type_lock=0&type_dropWorst=0&start={start}&num={length}&search="

DATA_DIRECTORY = f"data/{YEAR}"
GROUP_FILE = f'{DATA_DIRECTORY}/groups_{YEAR}.pkl'
OUTPUT_FILE_STUB = (
    f'{DATA_DIRECTORY}/espn_brackets_{YEAR}/'
    'espn_brackets_{}.json'
)
OUTPUT_FILENAME = f'{DATA_DIRECTORY}/espn_brackets_{YEAR}.feather'
SAVE_GAP = 10000

In [3]:
def get_espn_groups(length=100000):
    groups = []
    start = 0
    start_time = timeit.default_timer()
    while True:
        resp = requests.get(API_BASE + FIND_GROUP.format(start=start, length=length))
        data = json.loads(resp.text)["g"]
        if len(data) > 0:
            groups.extend([group["id"] for group in data])
            stop_time = timeit.default_timer()
            print(f"{start}: {stop_time - start_time}")
            start += length
        else:
            break
    return groups

In [4]:
%%time
groups = get_espn_groups()

0: 4.92704827
100000: 10.29885853899998
200000: 15.239493597999996
300000: 20.538142789999995
400000: 25.73143498799999
500000: 30.86414301299999
600000: 35.664687197000006
700000: 40.598907406999984
800000: 44.979902431
900000: 49.778096498999986
1000000: 54.355408495000006
1100000: 59.567712099999994
1200000: 64.46175985299999
1300000: 69.67777024
1400000: 74.126479523
1500000: 78.75231058699998
1600000: 81.20966078699999
CPU times: user 25.9 s, sys: 1.57 s, total: 27.5 s
Wall time: 1min 22s


In [5]:
len(groups)

1627340

In [6]:
groups = sorted(list(set(groups)))

In [7]:
pickle.dump(groups, open(GROUP_FILE, 'wb'))

In [19]:
def by_group(group_id):
    url = API_BASE + 'group?length=10000&groupID=' + str(group_id)
    d = {}
    start = 0
    while True:
        try:
            resp = requests.get(url + '&start=' + str(start))
        except requests.exceptions.RequestException as e:
            print("{}: {}".format(group_id, e))
            continue
        try:
            data = json.loads(resp.text)
            if 'g' not in data or 'e' not in data['g']:
                break
            else:
                ent = data['g']['e']
                start += len(ent)
                for e in ent:
                    d[e['id']] = e
        except Exception as e:
            print("{}: {}".format(group_id, e))
            continue
    return d


def groups_from_list(group_file, save_file_stub, save_gap):
    start_time = timeit.default_timer()

    groups = pickle.load(open(group_file, "rb"))

    i = 0
    for start in range(0, len(groups), save_gap):
        entries = {}
        stop = start + save_gap
        stop = len(groups) if stop > len(groups) else stop
        for group in groups[start:stop]:
            entries.update(by_group(group))
        with open(save_file_stub.format(i), 'w') as f:
            json.dump(entries, f)
        stop_time = timeit.default_timer()
        print(f"{stop}: {stop_time - start_time}")
        i += 1

In [20]:
%%time
groups_from_list(GROUP_FILE, OUTPUT_FILE_STUB, SAVE_GAP)

10000: 2894.521032855
555103: HTTPSConnectionPool(host='fantasy.espncdn.com', port=443): Max retries exceeded with url: /tournament-challenge-bracket/2022/en/api/group?length=10000&groupID=555103&start=9 (Caused by NewConnectionError('<urllib3.connection.VerifiedHTTPSConnection object at 0x16ca07fd0>: Failed to establish a new connection: [Errno 8] nodename nor servname provided, or not known'))
555103: HTTPSConnectionPool(host='fantasy.espncdn.com', port=443): Max retries exceeded with url: /tournament-challenge-bracket/2022/en/api/group?length=10000&groupID=555103&start=9 (Caused by NewConnectionError('<urllib3.connection.VerifiedHTTPSConnection object at 0x16ca07d90>: Failed to establish a new connection: [Errno 8] nodename nor servname provided, or not known'))
555103: HTTPSConnectionPool(host='fantasy.espncdn.com', port=443): Max retries exceeded with url: /tournament-challenge-bracket/2022/en/api/group?length=10000&groupID=555103&start=9 (Caused by NewConnectionError('<urllib3.co